# Explorateur des contrats de Montréal
Exécuter les cellules dans l’ordre. Version 1 : recherche paramétrée, classification indicative et export Excel. Aucun total de dépenses ni rapprochement automatique de contrats. Voir docs/explorateur_contrats_montreal.md.

In [ ]:
REF = 'main'  # Dernière version; ou SHA exact pour reproduire une analyse.
INCLURE = ['navette']
EXCLURE = []
MODE = 'OU'  # OU ou ET
CHAMP = 'tous'  # tous, fournisseur, description, decision, dossier, numero_source
SOURCES, SERVICES = [], []  # Listes vides : aucun filtre
ANNEE_MIN, ANNEE_MAX = None, None
MONTANT_MIN, MONTANT_MAX = None, None
CATEGORIE = ''  # Identifiant du dictionnaire; vide : toutes
EXCLURE_DOUBLONS_EXACTS = False


In [ ]:
import io, csv, json, hashlib, unicodedata
from datetime import datetime, timezone
import pandas as pd
import requests
from IPython.display import display

def normaliser(x):
    t = ''.join(c for c in unicodedata.normalize('NFKD', str(x)) if not unicodedata.combining(c)).upper()
    return ' '.join(''.join(c if c.isalnum() else ' ' for c in t).split())

def lire_csv(contenu):
    for enc in ['utf-8-sig', 'cp1252', 'latin-1']:
        try:
            texte = contenu.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    sep = csv.Sniffer().sniff(texte[:65536], delimiters=',;' + chr(9) + '|').delimiter
    return pd.read_csv(io.StringIO(texte), sep=sep, dtype=str, keep_default_na=False)

def harmoniser(df, fichier):
    def champ(*noms):
        for n in noms:
            if n in df:
                return df[n].astype(str)
        return pd.Series('', index=df.index)
    d = df.add_prefix('original__').copy()
    d['source'] = fichier
    d['ligne_source'] = df.index + 1
    d['fournisseur'] = champ('FOURNISSEUR', 'NOM DU FOURNISSEUR')
    d['description'] = champ('OBJET', 'DESCRIPTION')
    d['service'] = champ('SERVICE')
    d['direction'] = champ('DIRECTION')
    d['activite'] = champ('ACTIVITÉ')
    d['decision'] = champ('NO DÉCISION')
    d['dossier'] = champ('NO DE DOSSIER')
    d['numero_source'] = champ('NUMÉRO')
    d['date_originale'] = champ('DATE SIGNATURE', "DATE D'APPROBATION")
    d['type_date'] = 'signature' if 'DATE SIGNATURE' in df else 'approbation'
    # Pas de devinette jour/mois : les dates non ISO restent à vérifier.
    d['date'] = pd.to_datetime(d.date_originale.str.slice(0, 10), format='%Y-%m-%d', errors='coerce')
    d['annee'] = d.date.dt.year
    d['montant_original'] = champ('MONTANT')
    d['montant'] = pd.to_numeric(d.montant_original, errors='coerce')
    d['doublon_exact'] = df.duplicated(keep=False)
    d['empreinte_ligne'] = df.apply(lambda row: hashlib.sha256(json.dumps([fichier] + row.tolist(), ensure_ascii=False).encode()).hexdigest(), axis=1)
    d['description_index'] = d.description.map(normaliser)
    return d

def classer(d, categories, regles):
    if categories.categorie_id.duplicated().any() or regles.regle_id.duplicated().any():
        raise ValueError('Identifiants de configuration dupliqués')
    if not set(regles.categorie_id).issubset(set(categories.categorie_id)):
        raise ValueError('Catégorie inconnue dans les règles')
    preuves = []
    for r in regles.itertuples(index=False):
        expression, requis, exclu = map(normaliser, [r.expression, r.terme_requis, r.terme_exclu])
        if not expression:
            raise ValueError('Expression de règle vide')
        m = d.description_index.str.contains(expression, regex=False)
        if requis:
            m &= d.description_index.str.contains(requis, regex=False)
        if exclu:
            m &= ~d.description_index.str.contains(exclu, regex=False)
        p = d.loc[m, ['source', 'ligne_source', 'empreinte_ligne']].copy()
        p['regle_id'], p['categorie_id'], p['expression'] = r.regle_id, r.categorie_id, r.expression
        preuves.append(p)
    p = pd.concat(preuves, ignore_index=True).merge(categories, on='categorie_id', how='left')
    tags = p.groupby(['source', 'ligne_source']).categorie_id.agg(lambda s: '|'.join(sorted(set(s))))
    d = d.copy()
    d['categories'] = [tags.get((s, l), '') for s, l in zip(d.source, d.ligne_source)]
    return d, p

def rechercher(d):
    cibles = ['fournisseur', 'description', 'decision', 'dossier', 'numero_source'] if CHAMP == 'tous' else [CHAMP]
    texte = d[cibles].fillna('').agg(' | '.join, axis=1).map(normaliser)
    inclus, exclus = [normaliser(x) for x in INCLURE if x.strip()], [normaliser(x) for x in EXCLURE if x.strip()]
    m = pd.Series(True, index=d.index)
    if inclus:
        tests = pd.concat([texte.str.contains(x, regex=False) for x in inclus], axis=1)
        m &= tests.all(axis=1) if MODE == 'ET' else tests.any(axis=1)
    for x in exclus:
        m &= ~texte.str.contains(x, regex=False)
    for col, valeurs in [('source', SOURCES), ('service', SERVICES)]:
        if valeurs:
            m &= d[col].isin(valeurs)
    if ANNEE_MIN is not None:
        m &= d.annee.ge(ANNEE_MIN)
    if ANNEE_MAX is not None:
        m &= d.annee.le(ANNEE_MAX)
    if MONTANT_MIN is not None:
        m &= d.montant.ge(MONTANT_MIN)
    if MONTANT_MAX is not None:
        m &= d.montant.le(MONTANT_MAX)
    if CATEGORIE:
        m &= d.categories.str.split('|').map(lambda xs: CATEGORIE in xs)
    if EXCLURE_DOUBLONS_EXACTS:
        # Empreinte de contenu, pas un identifiant métier de contrat.
        m &= ~d.duplicated(['source', 'empreinte_ligne'])
    return d.loc[m].copy()


In [ ]:
OWNER, REPO = 'vince-fabmob', 'open_data_extractions'
r = requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/commits/{REF}', timeout=60)
r.raise_for_status()
SHA = r.json()['sha']
BASE = f'https://raw.githubusercontent.com/{OWNER}/{REPO}/{SHA}/'
def telecharger(chemin):
    r = requests.get(BASE + chemin, timeout=180)
    r.raise_for_status()
    return r.content
categories = lire_csv(telecharger('config/contrats_categories.csv'))
regles = lire_csv(telecharger('config/contrats_regles.csv'))
fichiers = ['contrats_comite_executif.csv', 'contrats_conseil_municipal.csv', 'contratsconseilsarrondissements.csv', 'contratsfonctionnaires.csv']
tables, qualite, manifeste_sources = [], [], []
for fichier in fichiers:
    contenu = telecharger(fichier)  # Un échec arrête l'analyse : pas de résultats partiels silencieux.
    brut = lire_csv(contenu)
    d = harmoniser(brut, fichier)
    tables.append(d)
    qualite.append({'source': fichier, 'lignes': len(d), 'doublons_supplementaires': int(brut.duplicated().sum()), 'dates_non_reconnues': int((d.date_originale.ne('') & d.date.isna()).sum()), 'montants_non_reconnus': int((d.montant_original.ne('') & d.montant.isna()).sum()), 'date_min': str(d.date.min()), 'date_max': str(d.date.max())})
    manifeste_sources.append({'fichier': fichier, 'sha256': hashlib.sha256(contenu).hexdigest()})
base, preuves = classer(pd.concat(tables, ignore_index=True).fillna({'description_index': ''}), categories, regles)
resultats = rechercher(base)
cles = resultats[['source', 'ligne_source']]
preuves_retenues = preuves.merge(cles, on=['source', 'ligne_source'], how='inner')
synthese = resultats.groupby(['source', 'annee'], dropna=False).size().rename('enregistrements').reset_index()
couverture = base.groupby(['source', 'annee'], dropna=False).size().rename('enregistrements').reset_index()
print('Commit :', SHA, '; résultats :', len(resultats), 'enregistrements, pas nécessairement des contrats uniques')
print('Classements proposés non validés. Aucun total de dépenses calculé.')
display(pd.DataFrame(qualite))
display(resultats[['source', 'fournisseur', 'description', 'date_originale', 'montant', 'categories']].head(100))
manifest = {'commit': SHA, 'execution_utc': datetime.now(timezone.utc).isoformat(), 'sources': manifeste_sources, 'parametres': {k: globals()[k] for k in ['REF', 'INCLURE', 'EXCLURE', 'MODE', 'CHAMP', 'SOURCES', 'SERVICES', 'ANNEE_MIN', 'ANNEE_MAX', 'MONTANT_MIN', 'MONTANT_MAX', 'CATEGORIE', 'EXCLURE_DOUBLONS_EXACTS']}}
# Neutraliser les formules dans les exports Excel, sans changer les données en mémoire.
def excel_sur(df):
    df = df.copy()
    for c in df.select_dtypes(include=['object', 'string']):
        df[c] = df[c].map(lambda v: "'" + v if isinstance(v, str) and v.startswith(('=', '+', '-', '@')) else v)
    return df
fichier_export = 'exploration_contrats.xlsx'
with pd.ExcelWriter(fichier_export, engine='openpyxl') as writer:
    for nom, table in [('Resultats', resultats.drop(columns=['description_index'])), ('Synthese', synthese), ('Preuves', preuves_retenues), ('Qualite', pd.DataFrame(qualite)), ('Couverture', couverture), ('Categories', categories), ('Regles', regles), ('Manifeste', pd.DataFrame({'json': [json.dumps(manifest, ensure_ascii=False)]}))]:
        if len(table) > 1048575:
            raise ValueError('Trop de lignes pour Excel; réduire la sélection')
        excel_sur(table).to_excel(writer, sheet_name=nom, index=False)
with open('exploration_manifeste.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
try:
    from google.colab import files
    files.download(fichier_export)
    files.download('exploration_manifeste.json')
except ImportError:
    print('Exports créés dans le dossier courant')
